# 裝飾器模式

## 學習目標

- 理解裝飾器的本質：函式包裝函式
- 會寫基本裝飾器、帶參數裝飾器
- 了解 `@functools.wraps` 的用途
- 掌握裝飾器堆疊順序
- 認識類別式裝飾器模式

---

## 1. 什麼是裝飾器？

裝飾器就是「接收一個函式，回傳一個新函式」的函式。用 `@` 語法把額外行為「包」在原函式外面，而不修改原函式的程式碼。

In [ ]:
def log(func):
    def wrapper(*args, **kwargs):
        print(f"呼叫 {func.__name__}，參數: {args}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} 回傳: {result}")
        return result
    return wrapper

@log
def add(a, b):
    return a + b

add(3, 5)
# 呼叫 add，參數: (3, 5)
# add 回傳: 8

`@log` 等同於 `add = log(add)`，只是語法糖。

---

## 2. functools.wraps：保留原函式資訊

沒有 `@wraps` 時，被裝飾的函式會「失去身份」：

In [ ]:
print(add.__name__)   # 印出 "wrapper"，不是 "add"
help(add)             # 顯示 wrapper 的文件，不是 add 的

加上 `@wraps` 就能保留原始資訊：

In [ ]:
import functools

def log(func):
    @functools.wraps(func)  # <-- 加這行
    def wrapper(*args, **kwargs):
        print(f"呼叫 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log
def add(a, b):
    """兩數相加"""
    return a + b

print(add.__name__)  # "add"
print(add.__doc__)   # "兩數相加"

> 規則：寫裝飾器時，永遠加 `@functools.wraps(func)`。

---

## 3. 帶參數的裝飾器

需要三層巢狀：外層接收參數，中層接收函式，內層執行邏輯。

In [ ]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"你好，{name}！")

greet("小明")
# 你好，小明！ （印三次）

呼叫順序：`repeat(3)` 回傳 `decorator`，`decorator(greet)` 回傳 `wrapper`。

---

## 4. 裝飾器堆疊順序

多個裝飾器由下往上包裝，由外往內執行：

In [ ]:
@decorator_A   # 第二層包裝（最外層）
@decorator_B   # 第一層包裝（最內層）
def my_func():
    pass

# 等同於：my_func = decorator_A(decorator_B(my_func))

執行時，A 的前置邏輯先跑，接著 B 的前置邏輯，然後原函式，再 B 的後置，最後 A 的後置。像洋蔥一樣一層一層。

---

## 5. 實用範例

### 計時裝飾器

In [ ]:
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} 耗時 {elapsed:.4f} 秒")
        return result
    return wrapper

@timer
def slow_task():
    time.sleep(1)
    return "完成"

### 快取（認識 lru_cache）

手寫快取很常見，但標準庫已經有現成的：

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(100)  # 瞬間完成，沒有重複計算

### 簡易權限檢查

In [ ]:
import functools

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user, *args, **kwargs):
            if user.get("role") != role:
                raise PermissionError(f"需要 {role} 權限")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@require_role("admin")
def delete_user(user, target_id):
    print(f"刪除用戶 {target_id}")

---

## 6. 類別式裝飾器模式（GoF Decorator）

函式裝飾器是 Python 特色，而物件導向的裝飾器模式核心概念是：透過組合而非繼承來動態增加功能。

```
Component（介面）
  +-- ConcreteComponent（原始物件）
  +-- Decorator（包裝基底）
        +-- ConcreteDecoratorA
        +-- ConcreteDecoratorB
```

### 咖啡店範例

In [ ]:
class Coffee:
    def cost(self):
        return 50  # 基本咖啡 50 元

class MilkDecorator:
    def __init__(self, coffee):
        self._coffee = coffee

    def cost(self):
        return self._coffee.cost() + 20

class SugarDecorator:
    def __init__(self, coffee):
        self._coffee = coffee

    def cost(self):
        return self._coffee.cost() + 5

order = SugarDecorator(MilkDecorator(Coffee()))
print(order.cost())  # 75 = 50 + 20 + 5

每一層「裝飾」都不修改原物件，而是包一層新功能。

---

## 總結

| 概念 | 重點 |
|------|------|
| 基本裝飾器 | 接收函式、回傳新函式 |
| `@functools.wraps` | 保留原函式名稱與文件，永遠要加 |
| 帶參數裝飾器 | 三層巢狀：參數 → 函式 → 執行 |
| 堆疊順序 | 下往上包裝，外往內執行（洋蔥模型） |
| 實用場景 | 計時、快取、權限檢查、日誌 |
| 類別式裝飾器 | 用組合動態添加功能，不改原物件 |

---

下一篇：[迭代器與生成器](02_iterator_and_generator.ipynb)